In [1]:
import pandas as pd
import glob
import os
from openpyxl.styles import Font

## Treino sem sampler e sem augmentation e sem segmentação

In [6]:
df_clean = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoSWeightSSeg\XAI\Clean\metricas_Clean.csv')
df_noise = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoSWeightSSeg\XAI\Noise\metricas_Noise.csv')
caminho_excel = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoSWeightSSeg\XAI\MetricasXAISWeightSSeg.xlsx'

## Treino com sampler e com augmentation e sem segmentação

In [9]:
df_clean = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightSSeg\XAI\Clean\metricas_Clean.csv')
df_noise = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightSSeg\XAI\Noise\metricas_Noise.csv')
caminho_excel = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightSSeg\XAI\MetricasXAICWeightSSeg.xlsx'

## Treino com sampler e com augmentation e com segmentação (fundo preto)

In [11]:
df_clean = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegPreto\XAI\Clean\metricas_Clean.csv')
df_noise = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegPreto\XAI\Noise\metricas_Noise.csv')
caminho_excel = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegPreto\XAI\MetricasXAICWeightCSegPreto.xlsx'

## Treino com sampler e com augmentation e com segmentação (fundo branco)

In [13]:
df_clean = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegBranco\XAI\Clean\metricas_Clean.csv')
df_noise = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegBranco\XAI\Noise\metricas_Noise.csv')
caminho_excel = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegBranco\XAI\MetricasXAICWeightCSegBranco.xlsx'

## Treino com sampler e com augmentation e com segmentação (fundo apenas ruído gaussiano)

In [15]:
df_clean = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegApenasRuido\XAI\Clean\metricas_Clean.csv')
df_noise = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegApenasRuido\XAI\Noise\metricas_Noise.csv')
caminho_excel = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegApenasRuido\XAI\MetricasXAICWeightCSegApenasRuido.xlsx'

## Treino com sampler e com augmentatin e com segmentação (fundo misto)

In [17]:
df_clean = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegMisto\XAI\Clean\metricas_Clean.csv')
df_noise = pd.read_csv(r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegMisto\XAI\Noise\metricas_Noise.csv')
caminho_excel = r'D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegMisto\XAI\MetricasXAICWeightCSegMisto.xlsx'

## Salvando as métricas

In [18]:
configs = {
    "Clean_Energy": (df_clean, "energy_inside"),
    "Clean_IoU": (df_clean, "iou"),
    "Noise_Energy": (df_noise, "energy_inside"),
    "Noise_IoU": (df_noise, "iou"),
}

with pd.ExcelWriter(caminho_excel, engine="openpyxl") as writer:

    for sheet, (df, metrica) in configs.items():
        
        row = 0

        pd.DataFrame(index=[]).to_excel(writer, sheet_name=sheet, index=False)
        
        worksheet = writer.sheets[sheet]
        
        # Global
        worksheet.cell(row=row+1, column=1).value = "GLOBAL"
        
        resultado = (df[metrica]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        
        resultado.columns = ["metric", "value"]
        
        resultado.to_excel(writer, sheet_name=sheet, startrow=row+1, index=False)

        row += len(resultado) + 4

        # By class
        worksheet.cell(row=row+1, column=1).value = "BY CLASS"
        
        resultado = (df.groupby("classe_real")[metrica]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        
        resultado.to_excel(writer, sheet_name=sheet, startrow=row+1, index=False)

        row += len(resultado) + 4

        # By method
        worksheet.cell(row=row+1, column=1).value = "BY METHOD"
        
        resultado = (df.groupby("metodo")[metrica]
            .agg(["mean", "std", "count"])
            .sort_values("mean", ascending=False)
            .reset_index()
        )
            
        resultado.to_excel(writer, sheet_name=sheet, startrow=row+1, index=False)

        row += len(resultado) + 4

        # By method and class
        worksheet.cell(row=row+1, column=1).value = "BY METHOD AND CLASS"
        
        resultado = (df.groupby(["metodo", "classe_real"])[metrica]
            .agg(["mean", "std", "count"])
            .sort_values(["classe_real", "mean"], ascending=[True, False])
            .reset_index() 
        )
        
        resultado.to_excel(writer, sheet_name=sheet, startrow=row+1, index=False)
        
        row += len(resultado) + 4

        # Correct prediction
        worksheet.cell(row=row+1, column=1).value = "BY CORRECT PREDICTION"
        
        resultado = (df.groupby("predicao_correta")[metrica]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        
        resultado.to_excel(writer, sheet_name=sheet, startrow=row+1, index=False)

## Métricas com todas as imagens do test set
Fiz um .csv file pra cada XAI method, então, preciso concatená-los antes. Além disso, os .csv já tem as duas métricas computadas.

## Treino sem sampler e sem augmentation e sem segmentação

In [28]:
CAMINHO_CSVS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoSWeightSSeg\XAI\FullTestSet"
caminho_excel = os.path.join(CAMINHO_CSVS, r"metricasXAIsWsS.xlsx")

## Treino com sampler e com augmentation e sem segmentação

In [2]:
CAMINHO_CSVS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightSSeg\XAI\FullTestSet"
caminho_excel = os.path.join(CAMINHO_CSVS, r"metricasXAIcWsS.xlsx")

## Treino com sampler e com augmentation e com segmentação (fundo preto)

In [5]:
CAMINHO_CSVS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegPreto\XAI\FullTestSet"
caminho_excel = os.path.join(CAMINHO_CSVS, r"metricasXAIcWcSPreto.xlsx")

## Treino com sampler e com augmentation e com segmentação (fundo branco)

In [8]:
CAMINHO_CSVS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegBranco\XAI\FullTestSet"
caminho_excel = os.path.join(CAMINHO_CSVS, r"metricasXAIcWcSBranco.xlsx")

## Treino com sampler e com augmentation e com segmentação (fundo apenas ruído gaussiano)

In [11]:
CAMINHO_CSVS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegApenasRuido\XAI\FullTestSet"
caminho_excel = os.path.join(CAMINHO_CSVS, r"metricasXAIcWcSRuido.xlsx")

## Treino com sampler e com augmentatin e com segmentação (fundo misto)

In [14]:
CAMINHO_CSVS = r"D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegMisto\XAI\FullTestSet"
caminho_excel = os.path.join(CAMINHO_CSVS, r"metricasXAIcWcSMisto.xlsx")

## Concatenado os .csv file

In [15]:
arquivos = glob.glob(
    os.path.join(CAMINHO_CSVS, "*.csv")
)

dfs = [pd.read_csv(arq) for arq in arquivos]

df_total = pd.concat(dfs, ignore_index=True)

print(df_total.shape)

caminho_csv = os.path.join(CAMINHO_CSVS, "metricasFull_Todas.csv")

df_total.to_csv(
    caminho_csv,
    index=False
)

(92708, 10)


In [16]:
# Load metrics CSV
df = pd.read_csv(caminho_csv)

# Split conditions
df_clean = df[df["condicao"] == "Clean"]
df_noise = df[df["condicao"] == "Noise"]

configs = {
    "Clean_Energy": (df_clean, "energy_inside"),
    "Clean_IoU": (df_clean, "iou"),
    "Noise_Energy": (df_noise, "energy_inside"),
    "Noise_IoU": (df_noise, "iou"),
}

with pd.ExcelWriter(caminho_excel, engine="openpyxl") as writer:

    for sheet, (df_base, metrica) in configs.items():

        row = 0

        # Create sheet
        pd.DataFrame(index=[]).to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )

        worksheet = writer.sheets[sheet]

        # =====================================================
        # GLOBAL
        # =====================================================

        worksheet.cell(row=row + 1, column=1).value = "GLOBAL"
        worksheet.cell(row=row + 1, column=1).font = Font(bold=True)

        resultado = (
            df_base[metrica]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        resultado.columns = ["metric", "value"]

        resultado.to_excel(
            writer,
            sheet_name=sheet,
            startrow=row + 1,
            index=False
        )

        row += len(resultado) + 6

        # =====================================================
        # BY CLASS
        # =====================================================

        worksheet.cell(row=row + 1, column=1).value = "BY CLASS"
        worksheet.cell(row=row + 1, column=1).font = Font(bold=True)

        resultado = (
            df_base.groupby("classe_real")[metrica]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        resultado.to_excel(
            writer,
            sheet_name=sheet,
            startrow=row + 1,
            index=False
        )

        row += len(resultado) + 6

        # =====================================================
        # BY METHOD
        # =====================================================

        worksheet.cell(row=row + 1, column=1).value = "BY METHOD"
        worksheet.cell(row=row + 1, column=1).font = Font(bold=True)

        resultado = (
            df_base.groupby("metodo")[metrica]
            .agg(["mean", "std", "count"])
            .sort_values("mean", ascending=False)
            .reset_index()
        )

        resultado.to_excel(
            writer,
            sheet_name=sheet,
            startrow=row + 1,
            index=False
        )

        row += len(resultado) + 6

        # =====================================================
        # BY METHOD AND CLASS
        # =====================================================

        worksheet.cell(row=row + 1, column=1).value = "BY METHOD AND CLASS"
        worksheet.cell(row=row + 1, column=1).font = Font(bold=True)

        resultado = (
            df_base.groupby(["metodo", "classe_real"])[metrica]
            .agg(["mean", "std", "count"])
            .sort_values(
                ["classe_real", "mean"],
                ascending=[True, False]
            )
            .reset_index()
        )

        resultado.to_excel(
            writer,
            sheet_name=sheet,
            startrow=row + 1,
            index=False
        )

        row += len(resultado) + 6

        # =====================================================
        # BY CORRECT PREDICTION
        # =====================================================

        worksheet.cell(row=row + 1, column=1).value = "BY CORRECT PREDICTION"
        worksheet.cell(row=row + 1, column=1).font = Font(bold=True)

        resultado = (
            df_base.groupby("predicao_correta")[metrica]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        resultado.to_excel(
            writer,
            sheet_name=sheet,
            startrow=row + 1,
            index=False
        )

        row += len(resultado) + 6

        # =====================================================
        # BY METHOD AND CORRECT PREDICTION
        # =====================================================

        worksheet.cell(row=row + 1, column=1).value = "BY METHOD AND CORRECT PREDICTION"
        worksheet.cell(row=row + 1, column=1).font = Font(bold=True)

        resultado = (
            df_base.groupby(
                ["metodo", "predicao_correta"]
            )[metrica]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        resultado.to_excel(
            writer,
            sheet_name=sheet,
            startrow=row + 1,
            index=False
        )

print(f"Excel saved to: {caminho_excel}")

Excel saved to: D:\Analise_Esporos_2026\Modelo_Classificacao_4_classes\TreinoParalelo\TreinoCWeightCSegMisto\XAI\FullTestSet\metricasXAIcWcSMisto.xlsx
